# Yemberzal Kashmiri TTS listening test
Research-only evaluation of the existing Kashmiri Svara/SNAC checkpoint. This notebook does not publish or deploy the model.

In [ ]:
!nvidia-smi
!pip install -q 'transformers>=4.48,<5' accelerate snac soundfile safetensors

In [ ]:
import json, torch, soundfile as sf
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from snac import SNAC
MODEL_ID = 'sandysidhu12/yemberzal-tts-kashmiri'
OUT = Path('/kaggle/working/yemberzal-kashmiri-samples'); OUT.mkdir(exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', low_cpu_mem_usage=True).eval()
codec = SNAC.from_pretrained('hubertsiuzdak/snac_24khz').eval().to('cpu')
print('Models loaded', flush=True)

In [ ]:
BOS, EOT, AUDIO = 128000, 128009, 156939
SOS, EOS, SOH, EOH, SOA = 128257, 128258, 128259, 128260, 128261
OFFSETS = [128266 + i * 4096 for i in range(7)]
def prompt_ids(text):
    words = tokenizer(f'Kashmiri (Male): {text}', add_special_tokens=False).input_ids
    return [BOS, SOH, AUDIO, *words, EOH, EOT, SOA, SOS]
def audio_codes(ids):
    result = []
    for token in ids:
        if token == EOS: break
        offset = OFFSETS[len(result) % 7]
        code = int(token) - offset
        if 0 <= code < 4096: result.append(code)
    return result[:len(result) - (len(result) % 7)]
def decode(codes):
    frames = torch.tensor(codes, dtype=torch.long).view(-1, 7)
    streams = [frames[:,0].view(1,-1), frames[:,[1,4]].reshape(1,-1), frames[:,[2,3,5,6]].reshape(1,-1)]
    with torch.inference_mode(): return codec.decode(streams).squeeze().float().cpu().numpy()
texts = ['کٲشُر زَبان چھِ اَکھ خوٗبصوٗرَت زَبان۔', 'عِلم چھُ سارِوٕے کھۄتہٕ بٔڑ دَولَتھ۔', 'اَسَلامُ عَلَیکُم، تُہۍ کِتھ پٲٹھۍ چھِو؟']
report = []
device = next(model.parameters()).device
for index, text in enumerate(texts, 1):
    inputs = torch.tensor([prompt_ids(text)], dtype=torch.long, device=device)
    with torch.inference_mode():
        generated = model.generate(inputs, max_new_tokens=1400, do_sample=True, temperature=0.6, top_p=0.9, repetition_penalty=1.1, eos_token_id=EOS, pad_token_id=tokenizer.eos_token_id)
    codes = audio_codes(generated[0, inputs.shape[1]:].tolist())
    assert len(codes) >= 28, f'No valid audio tokens for sample {index}'
    audio = decode(codes)
    path = OUT / f'sample-{index}.wav'; sf.write(path, audio, 24000)
    report.append({'text':text, 'audio':str(path), 'codes':len(codes), 'seconds':round(len(audio)/24000,2)})
    print(report[-1], flush=True)
(OUT/'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))
print('Listening samples ready:', OUT)